In [1]:
#!/usr/bin/env python3
"""The sliding-tile puzzle itself: state, moves, and the solvability constraint.

Nothing in here knows about search. A state is a plain tuple of n*n ints, row
major, with 0 for the blank:

    (5, 1, 2, 3,          5  1  2  3
     4, 6, 8, 7,     ==   4  6  8  7
     9, 0, 10, 11,        9  .  10 11
     12, 13, 14, 15)      12 13 14 15

Tuples are hashable and immutable, so they drop straight into the `set` of
closed nodes and the `dict` of g-values that A* needs. That is the whole reason
for the representation.

THE CONSTRAINT (why half of all boards are unsolvable)
-----------------------------------------------------
Every legal move swaps the blank with a neighbour. Two things happen at once:

  1. the permutation of the 16 cells is multiplied by a transposition, so its
     parity flips;
  2. the blank moves one step, so the parity of its taxicab distance from home
     flips too.

Their sum is therefore invariant. The goal has sum 0 (even), so:

    a state is solvable  <=>  perm_parity(state) + blank_taxicab(state) is even

That is exact for any board size and, unlike the textbook "count inversions"
rule, for any goal - including the deck's goal with the blank in the CORNER.
`Puzzle.solvable()` is the only gate; every constructor here goes through it.
"""
from __future__ import annotations

import random

# blank moves; (row delta, col delta, name). The name describes where the BLANK
# goes, so a solution path reads as a sequence of blank moves.
MOVES = ((-1, 0, "U"), (1, 0, "D"), (0, -1, "L"), (0, 1, "R"))
OPPOSITE = {"U": "D", "D": "U", "L": "R", "R": "L"}


class Puzzle:
    """An n x n sliding-tile problem: a goal, the moves, and the constraint.

    n = 3 gives the 8-puzzle of the slides, n = 4 the 15-puzzle. Nothing below
    is hard-coded to either.
    """

    def __init__(self, size: int = 4, goal: tuple[int, ...] | None = None):
        if size < 2:
            raise ValueError("size must be at least 2")
        self.n = size
        self.cells = size * size
        # default goal: blank in the TOP-LEFT corner, 1..n^2-1 after it. This is
        # the goal on slide 36 of eai-01-search (_ 1 2 / 3 4 5 / 6 7 8), not the more
        # common blank-last convention. Pass `goal=` for the other one.
        self.goal = tuple(goal) if goal else tuple(range(self.cells))
        self.validate(self.goal)
        # goal_pos[tile] = (row, col) it belongs in - used by every heuristic
        self.goal_pos = [(0, 0)] * self.cells
        for i, tile in enumerate(self.goal):
            self.goal_pos[tile] = divmod(i, self.n)
        self.blank_home = self.goal_pos[0]

    # ---------------------------------------------------------------- validity

    def validate(self, state) -> tuple[int, ...]:
        """Reject anything that is not a permutation of 0..n^2-1."""
        state = tuple(state)
        if len(state) != self.cells or sorted(state) != list(range(self.cells)):
            raise ValueError(
                f"a {self.n}x{self.n} board must be a permutation of "
                f"0..{self.cells - 1}, got {state}"
            )
        return state

    def perm_parity(self, state) -> int:
        """0 if the permutation state -> goal is even, 1 if odd.

        Counted by cycle decomposition, which is O(n^2) and, unlike inversion
        counting, needs no assumption about where the blank lives in the goal.
        """
        where = {tile: i for i, tile in enumerate(state)}
        target = [where[tile] for tile in self.goal]  # target[i] = index in state
        seen = [False] * self.cells
        parity = 0
        for i in range(self.cells):
            if seen[i]:
                continue
            length = 0
            j = i
            while not seen[j]:
                seen[j] = True
                j = target[j]
                length += 1
            parity ^= (length - 1) & 1  # a k-cycle is k-1 transpositions
        return parity

    def blank_taxicab(self, state) -> int:
        r, c = divmod(state.index(0), self.n)
        gr, gc = self.blank_home
        return abs(r - gr) + abs(c - gc)

    def solvable(self, state) -> bool:
        """The invariant above. Half of all boards fail it - by construction."""
        state = self.validate(state)
        return (self.perm_parity(state) + self.blank_taxicab(state)) % 2 == 0

    def require_solvable(self, state) -> tuple[int, ...]:
        state = self.validate(state)
        if not self.solvable(state):
            raise ValueError(
                "this board is in the OTHER orbit - no sequence of moves reaches "
                "the goal. Swap any two non-blank tiles to fix it.\n"
                + self.render(state)
            )
        return state

    # ------------------------------------------------------------------- moves

    def neighbors(self, state):
        """Yield (next_state, move) for each legal blank move. Every move costs 1."""
        blank = state.index(0)
        br, bc = divmod(blank, self.n)
        for dr, dc, name in MOVES:
            r, c = br + dr, bc + dc
            if 0 <= r < self.n and 0 <= c < self.n:
                swap = r * self.n + c
                nxt = list(state)
                nxt[blank], nxt[swap] = nxt[swap], 0
                yield tuple(nxt), name

    def apply(self, state, move: str):
        for nxt, name in self.neighbors(state):
            if name == move:
                return nxt
        raise ValueError(f"move {move!r} is illegal from\n{self.render(state)}")

    def is_goal(self, state) -> bool:
        return tuple(state) == self.goal

    # -------------------------------------------------------------- generation

    def scramble(self, moves: int = 40, seed: int | None = None):
        """A random WALK back from the goal - solvable by construction.

        `moves` is an upper bound on the optimal solution length, not the
        optimal length itself: the walk undoes itself sometimes. It is the knob
        that keeps a classroom demo finishing before the bell.
        """
        rng = random.Random(seed)
        state, last = self.goal, None
        for _ in range(moves):
            options = [(s, m) for s, m in self.neighbors(state)
                       if last is None or m != OPPOSITE[last]]
            state, last = rng.choice(options)
        return state

    def random_state(self, seed: int | None = None):
        """A uniformly random SOLVABLE board - i.e. the hard end of the pool.

        Shuffle, then repair parity by swapping two non-blank tiles. For the
        15-puzzle these average ~53 moves to solve, which is out of reach for
        plain A* and is exactly why `idastar.py` exists.
        """
        rng = random.Random(seed)
        state = list(range(self.cells))
        rng.shuffle(state)
        if not self.solvable(tuple(state)):
            i, j = [k for k, t in enumerate(state) if t != 0][:2]
            state[i], state[j] = state[j], state[i]
        return tuple(state)

    # ------------------------------------------------------------------- text

    def parse(self, text: str):
        """Read "7 2 4 / 5 _ 6 / 8 3 1" (or newlines, or 0 for the blank)."""
        tokens = text.replace("/", " ").replace("|", " ").split()
        state = tuple(0 if t in ("_", ".", "0", "x") else int(t) for t in tokens)
        return self.validate(state)

    def render(self, state, cell: int | None = None) -> str:
        w = cell or len(str(self.cells - 1))
        rows = []
        for r in range(self.n):
            row = state[r * self.n:(r + 1) * self.n]
            rows.append(" ".join(("_" if t == 0 else str(t)).rjust(w) for t in row))
        return "\n".join(rows)

    def render_path(self, state, moves) -> str:
        """Every board along a solution, side by side, wrapped to the terminal."""
        boards = [self.render(state)]
        labels = ["start"]
        for m in moves:
            state = self.apply(state, m)
            boards.append(self.render(state))
            labels.append(m)
        out, per_line = [], max(1, 76 // (self.n * (len(str(self.cells - 1)) + 1) + 4))
        for i in range(0, len(boards), per_line):
            chunk, tags = boards[i:i + per_line], labels[i:i + per_line]
            width = max(len(l.split("\n")[0]) for l in chunk) + 4
            out.append("".join(t.center(width) for t in tags))
            for r in range(self.n):
                out.append("".join(b.split("\n")[r].center(width) for b in chunk))
            out.append("")
        return "\n".join(out)

        # ------------------------------------------------------------------ search

    def manhattan_distance(self, state: tuple[int, ...]):
        distance = 0
        #iterate over all tiles in the current board
        for i, tile in enumerate(state):
            if tile != 0:  #ignores the blank tile
                row, column = divmod(i, self.n) #converts 1d state index to 2d row/column coordinates (used google to help with this part)
                goal_row, goal_column = self.goal_pos[tile] #lookup the goal position of specific tile in goal_pos
                distance += abs(row - goal_row) + abs(column - goal_column) #accumlate the total distance for this specific tile
        return distance

    def greedy_search(self, start_state: tuple[int, ...]):
        import heapq

        #makes sure the puzzle is solvability as to not waste resources
        start = self.require_solvable(start_state)

        start_heuristic = self.manhattan_distance(start)

        #priority queue ordered by the heuristic, and in total that and the state and path
        pq = [(start_heuristic, start, [])]

        closed_set = set()

        #stats
        expanded = 0
        generated = 1
        peak_frontier = 1

        while pq:
            #pop node with lowest estimated distance to the heuristic to the goal
            h, current, path = heapq.heappop(pq)

            if current in closed_set:
                continue
            closed_set.add(current)
            expanded += 1

            if self.is_goal(current):
                return path, expanded, generated, peak_frontier

            #expand successor states
            for next_state, move in self.neighbors(current):
                if next_state not in closed_set:
                    next_h = self.manhattan_distance(next_state)
                    #push the neighbor that is solely picked based upon the heuristic
                    heapq.heappush(pq, (next_h, next_state, path + [move]))
                    generated += 1

                #maximum memory footprint of the prioirty queue
                peak_frontier = max(peak_frontier, len(pq))

        return None, expanded, generated, peak_frontier

    def astar(self, start_state: tuple[int, ...]):
        import heapq

        #make sure solvable
        start = self.require_solvable(start_state)

        #add g to the prioirty queue so can track the traveled distance already
        start_heuristic = self.manhattan_distance(start)
        #priority queue is ordered by f_score primarily because in tuple it is considered to be first, and goes further if needed
        pq = [(start_heuristic, 0, start, [])]

        #no travelled distance yet before we start
        g_scores = {start: 0}

        #makes sure we do not visit same locations multiple times
        closed_set = set()
        # stats
        expanded = 0
        generated = 1
        peak_frontier = 1

        #loops as long as unvisited member remains available in the p queue
        while pq:
            #pops lowest f_score (or closest to goal) into variables left to right
            f, g, current, path = heapq.heappop(pq)

            #if shorter path has already been found to current positions
            if g > g_scores.get(current, float("inf")):
                continue

            expanded += 1

            if self.is_goal(current):
                return path, expanded, generated, peak_frontier
            
            #expand all neighbors
            for next_state, move in self.neighbors(current):
                #path costs add  1 to the new g
                new_g = g + 1

                #pick the neighbor if based on if a shorter path exists then the one currently stored
                if new_g < g_scores.get(next_state, float("inf")):
                    g_scores[next_state] = new_g
                    #next herustic is found
                    next_h = self.manhattan_distance(next_state)
                    f_score = new_g + next_h
                    #enqueue neighbor with updated costs and move list
                    heapq.heappush(pq, (f_score, new_g, next_state, path + [move]))
                    generated += 1

            peak_frontier = max(peak_frontier, len(pq))
        return None, expanded, generated, peak_frontier


# The two boards that appear in the deck (session-01/eai-01-search.md, slides 20 and 36).
def reversed_board(size: int = 4, blank_last: bool = True) -> tuple[int, ...]:
    """Every tile in reverse order - which is impossible on an even board.

        4x4:  15 14 13 12 / 11 10 9 8 / 7 6 5 4 / 3 2 1 _     no solution exists
        3x3:   8  7  6 /  5  4  3 /  2  1 _                    solvable in 30 moves

    It is the most convincing unsolvable board to show a class, because nothing
    about it looks wrong: it is perfectly ordered, just backwards.

    The reason is the parity invariant and nothing else. Reversing n^2-1 tiles is
    (n^2-1)//2 swaps, and the blank never leaves home, so

        3x3:  8 tiles ->  4 swaps -> even -> reachable
        4x4: 15 tiles ->  7 swaps -> ODD  -> unreachable
        5x5: 24 tiles -> 12 swaps -> even -> reachable
        6x6: 35 tiles -> 17 swaps -> ODD  -> unreachable

    So the same idea - "put them all backwards" - flips between possible and
    impossible as the board grows. Half of all boards are unreachable, and this
    is a member of that half you can write down from memory.
    """
    tiles = list(range(1, size * size))[::-1]
    return tuple(tiles + [0]) if blank_last else tuple([0] + tiles)


# slide 20, the DFS/BFS trees - with the spiral goal that board is posed against
EIGHT_PUZZLE_DFS = ("2 8 3 / 1 6 4 / 7 _ 5",
                    "1 2 3 / 8 _ 4 / 7 6 5")
# slide 36, the A* board
EIGHT_PUZZLE_ASTAR = ("7 2 4 / 5 _ 6 / 8 3 1",
                      "_ 1 2 / 3 4 5 / 6 7 8")


# if __name__ == "__main__":
#     # Setup 3x3 puzzle using slide 36 boards from the docstring
#     p = Puzzle(3, goal=Puzzle(3).parse(EIGHT_PUZZLE_ASTAR[1]))
#     start_state = p.parse(EIGHT_PUZZLE_ASTAR[0])

# if __name__ == "__main__":

#     puzzle_strings = [
#     "1 2 7 3 / 5 6 _ 4 / 9 10 11 8 / 13 14 15 12",
#     "1 6 _ 4 / 5 3 2 8 / 9 15 7 11 / 13 10 14 12",
#     "5 1 11 8 / 9 7 2 3 / 10 14 4 6 / 13 _ 15 12",
#     "_ 2 4 8 / 1 7 3 6 / 10 5 11 12 / 9 14 13 15",
#     "6 11 2 3 / 9 _ 5 10 / 13 1 15 4 / 14 8 12 7"
#     ]   

#     standard_goal = tuple(range(1,16)) + (0,)
#     p = Puzzle(4, goal=standard_goal)

#     for i, p_str in enumerate(puzzle_strings, start=1):
#         start_state = p.parse(p_str)
#         is_solvable = p.solvable(start_state)
        
#         print("==START BOARD ===")
#         print(p.render(start_state))
#         print(f"\nSolvable: {p.solvable(start_state)}\n")

#         # 1. Run Greedy Search
#         greedy_path = p.greedy_search(start_state)
#         print("=== GREEDY BEST-FIRST SEARCH ===")
#         if greedy_path is not None:
#             print(f"Path Length : {len(greedy_path)} moves")
#             print(f"Moves       : {' -> '.join(greedy_path)}\n")
#         else:
#             print("No solution found.\n")
        
#         # 2. Run A* Search
#         astar_path = p.astar(start_state)
#         print("=== A* SEARCH (OPTIMAL) ===")
#         print(f"Path Length : {len(astar_path)} moves")
#         print(f"Moves       : {' -> '.join(astar_path)}\n")

if __name__ == "__main__":
    import json
    import time

    #example test cases of start states listed in the proper format for the board
    puzzle_strings = [
        "1 2 7 3 / 5 6 _ 4 / 9 10 11 8 / 13 14 15 12",
        "1 6 _ 4 / 5 3 2 8 / 9 15 7 11 / 13 10 14 12",
        "5 1 11 8 / 9 7 2 3 / 10 14 4 6 / 13 _ 15 12",
        "_ 2 4 8 / 1 7 3 6 / 10 5 11 12 / 9 14 13 15",
        "6 11 2 3 / 9 _ 5 10 / 13 1 15 4 / 14 8 12 7"
    ]   

    #the formatted properly goal string of the board
    standard_goal_str = "1 2 3 4 / 5 6 7 8 / 9 10 11 12 / 13 14 15 _"
    #goal listed as a tuple that goes 1-15 and is followed by 0 as the final, bottom right tile
    standard_goal = tuple(range(1, 16)) + (0,)
    p = Puzzle(4, goal=standard_goal) #initializes the 4x4 board

    #accumlator for structured test run metrics as a list
    json_records = []

    #run through the example puzzles
    for i, p_str in enumerate(puzzle_strings, start=1):
        #convert the string format into the flat coordinate tuple format
        start_state = p.parse(p_str)

        # Run A* Search (used google to help)
        t0 = time.perf_counter()
        astar_path, a_exp, a_gen, a_peak = p.astar(start_state)
        astar_sec = time.perf_counter() - t0
        #store the minimum path costs
        optimal_cost = len(astar_path) if astar_path else None

        # Run Greedy Search
        t0 = time.perf_counter()
        greedy_path, g_exp, g_gen, g_peak = p.greedy_search(start_state)
        greedy_sec = time.perf_counter() - t0

        #group the searches so it is easier for uniform processing
        runs = [
            ("greedy", greedy_path, g_exp, g_gen, g_peak, greedy_sec),
            ("astar", astar_path, a_exp, a_gen, a_peak, astar_sec)
        ]

        #standardize search metrics into submission schema format
        for name, path, exp, gen, peak, runtime in runs:
            found = path is not None
            cost = len(path) if found else None
            is_optimal = "yes" if found and cost == optimal_cost else "no"

            #construct standardized data payload entry
            record = {
                "board_id": f"board_{i}",
                "algorithm": name,
                "start": p_str,
                "goal": standard_goal_str,
                "moves_are": "blank",
                "moves": path if found else [],
                "found": found,
                "solution_length": cost,
                "solution_cost": cost,
                "expanded": exp,
                "generated": gen,
                "peak_frontier": peak,
                "runtime_seconds": runtime,
                "cutoff": None,
                "optimality": is_optimal,
                "verified": found
            }
            json_records.append(record)

    #wrap submission records inside top level evelope schema
    submission_payload = {
        "assignment": "homework-01",
        "puzzle_results": json_records
    }

    #export benchmark output payload to target JSON formatted properly (used google to help me figure out schema, how to export)
    with open("puzzle-solutions.json", "w") as f:
        json.dump(submission_payload, f, indent=2)

    print("Saved 10 puzzle results to puzzle-solutions.json successfully.")

Saved 10 puzzle results to puzzle-solutions.json successfully.


In [2]:
#!/usr/bin/env python3
"""Validate the standardized Homework 01 puzzle submission.

    python3 check_submission.py path/to/puzzle-solutions.json

The checker validates the submission record and replays every reported
successful solution. It is intentionally a format checker, not a grader.
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

from board import Puzzle
from verify import check, parse_moves

REQUIRED = {
    "board_id", "algorithm", "start", "goal", "moves_are", "moves", "found",
    "solution_length", "solution_cost", "expanded", "generated", "peak_frontier",
    "runtime_seconds", "cutoff", "optimality", "verified",
}
ALGORITHMS = {"dfs", "bfs", "greedy", "astar", "ucs", "iddfs", "idastar", "other"}


def fail(label: str, message: str):
    raise ValueError(f"{label}: {message}")


def number(value, label: str, *, integer=False):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value < 0:
        fail(label, "must be a non-negative number")
    if integer and not isinstance(value, int):
        fail(label, "must be a non-negative integer")


def standard_board_text(text: str, label: str) -> int:
    """Require the course notation: rows separated by / and _ for the blank."""
    if not isinstance(text, str):
        fail(label, "must be a board-state string")
    rows = [row.strip() for row in text.split("/")]
    if len(rows) < 2 or any(not row for row in rows):
        fail(label, "must use / to separate rows")
    size = len(rows)
    tokens = [row.split() for row in rows]
    if any(len(row) != size for row in tokens):
        fail(label, f"must have {size} tiles in each of its {size} rows")
    flat = [tile for row in tokens for tile in row]
    if flat.count("_") != 1:
        fail(label, "must use exactly one _ for the blank (not 0 or .)")
    if any(tile in {"0", ".", "x"} for tile in flat):
        fail(label, "must use _ for the blank (not 0, ., or x)")
    return size


def puzzle_for(record: dict, label: str) -> tuple[Puzzle, tuple[int, ...]]:
    size = standard_board_text(record["start"], f"{label}.start")
    goal_size = standard_board_text(record["goal"], f"{label}.goal")
    if goal_size != size:
        fail(label, "start and goal must have the same board size")
    scratch = Puzzle(size)
    try:
        goal = scratch.parse(record["goal"])
        puzzle = Puzzle(size, goal=goal)
        return puzzle, puzzle.require_solvable(puzzle.parse(record["start"]))
    except ValueError as error:
        fail(label, str(error))


def validate_record(record, index: int):
    label = f"puzzle_results[{index}]"
    if not isinstance(record, dict):
        fail(label, "must be an object")
    missing = REQUIRED - record.keys()
    if missing:
        fail(label, "missing " + ", ".join(sorted(missing)))
    if not isinstance(record["board_id"], str) or not record["board_id"].strip():
        fail(label, "board_id must be a non-empty string")
    if record["algorithm"] not in ALGORITHMS:
        fail(label, "algorithm must be one of " + ", ".join(sorted(ALGORITHMS)))
    if record["moves_are"] != "blank":
        fail(label, "moves_are must be 'blank'")
    if not isinstance(record["moves"], list) or not all(isinstance(m, str) for m in record["moves"]):
        fail(label, "moves must be an array of move strings")
    if not isinstance(record["found"], bool) or not isinstance(record["verified"], bool):
        fail(label, "found and verified must be true or false")
    if record["optimality"] not in {"yes", "no", "unknown"}:
        fail(label, "optimality must be yes, no, or unknown")
    if record["cutoff"] is not None and not isinstance(record["cutoff"], str):
        fail(label, "cutoff must be a string or null")
    for field in ("expanded", "generated", "peak_frontier"):
        number(record[field], f"{label}.{field}", integer=True)
    number(record["runtime_seconds"], f"{label}.runtime_seconds")

    puzzle, start = puzzle_for(record, label)
    try:
        moves = parse_moves(" ".join(record["moves"]))
    except ValueError as error:
        fail(label, str(error))
    state, failed_at, _ = check(puzzle, start, moves)
    solved = failed_at is None and puzzle.is_goal(state)

    if record["found"]:
        if not solved:
            fail(label, "found is true but the moves do not reach the stated goal")
        if not record["verified"]:
            fail(label, "a found solution must have verified: true")
        for field in ("solution_length", "solution_cost"):
            if record[field] != len(moves):
                fail(label, f"{field} must equal the {len(moves)} reported moves")
    else:
        if moves or record["verified"]:
            fail(label, "an unsuccessful run must use no moves and verified: false")
        if record["solution_length"] is not None or record["solution_cost"] is not None:
            fail(label, "an unsuccessful run must use null solution_length and solution_cost")


def main(argv=None):
    argv = sys.argv[1:] if argv is None else argv
    if len(argv) != 1:
        print("usage: python3 check_submission.py puzzle-solutions.json", file=sys.stderr)
        return 2
    try:
        data = json.loads(Path(argv[0]).read_text())
        if not isinstance(data, dict) or data.get("assignment") != "homework-01":
            fail("submission", "assignment must be 'homework-01'")
        records = data.get("puzzle_results")
        if not isinstance(records, list) or not records:
            fail("submission", "puzzle_results must be a non-empty array")
        for index, record in enumerate(records):
            validate_record(record, index)
    except (OSError, json.JSONDecodeError, ValueError) as error:
        print(f"INVALID: {error}", file=sys.stderr)
        return 1
    print(f"VALID: {len(records)} puzzle result(s) checked.")
    return 0


# At the bottom of Cell 21, replace:
# if __name__ == "__main__":
#     raise SystemExit(main())

# With this:
if __name__ == "__main__":
    main(["puzzle-solutions.json"])

VALID: 10 puzzle result(s) checked.
